# 04 — Multi-Source Data Audit

**Phase 3 objective:** profile the five gaming datasets, compare their grains and fields, and generate candidate cross-source matches without automatically merging incompatible measures.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# Project paths
# ============================================================

# Expected project structure:
#
# Game_Launch_Intelligence/
# ├── Game_Data/
# ├── Python_Analysis/
# │   └── notebooks/
# └── SQL_Analysis/

ROOT = Path.cwd().parents[1]

DATA = ROOT / "Game_Data"


# ============================================================
# Source datasets
# ============================================================

source_paths = {
    "VGChartz":
        DATA / "vgchartz-2024.csv",

    "Video Games Sales":
        DATA / "Video_Games(1).csv",

    "Steam Store":
        DATA / "steam-games(1).csv",

    "SteamSpy":
        DATA / "steamspy_data.csv",

    "Steam Cleaned 2026":
        DATA / "steam_cleaned_2026.csv",
}


# ============================================================
# Validate source files
# ============================================================

print("=" * 70)
print("SOURCE DATA VALIDATION")
print("=" * 70)

for name, path in source_paths.items():
    print(f"{name:<22}: {path} -> {path.exists()}")

print("=" * 70)

SOURCE DATA VALIDATION
VGChartz              : c:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Game_Data\vgchartz-2024.csv -> True
Video Games Sales     : c:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Game_Data\Video_Games(1).csv -> True
Steam Store           : c:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Game_Data\steam-games(1).csv -> True
SteamSpy              : c:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Game_Data\steamspy_data.csv -> True
Steam Cleaned 2026    : c:\Users\USER\Desktop\Resume Projects\Game_Launch_Intelligence\Game_Data\steam_cleaned_2026.csv -> True


In [8]:
def load_sample(path, nrows=1000):
    return pd.read_csv(path, nrows=nrows, low_memory=False)

profiles=[]
for source,path in source_paths.items():
    if path.exists():
        df=load_sample(path)
        profiles.append({
            'source': source,
            'sample_rows': len(df),
            'columns': len(df.columns),
            'fields': ', '.join(df.columns)
        })

pd.DataFrame(profiles)


,source,sample_rows,columns,fields
0,VGChartz,1000,14,"img, title, console, genre, publisher, develop..."
1,Video Games Sales,1000,17,"index, Name, Platform, Year_of_Release, Genre,..."
2,Steam Store,1000,12,"title, genres, description, url, price, discou..."
3,SteamSpy,1000,20,"appid, name, developer, publisher, score_rank,..."
4,Steam Cleaned 2026,1000,38,"AppID, Name, Release date, Estimated owners, P..."


## Grain and measurement rules

The five datasets are deliberately not unioned. Console sales, Steam estimated ownership, reviews, CCU, pricing and playtime are retained as separate measures. Cross-source linkage is an identity problem, not a sales aggregation problem.

In [9]:
def norm_text(x):
    x='' if pd.isna(x) else str(x).lower()
    x=re.sub(r'[^a-z0-9]+',' ',x).strip()
    return re.sub(r'\s+',' ',x)

# Example candidate-key generator for sources that contain developer/publisher/year.
def candidate_key(df, title_col, developer_col=None, publisher_col=None, year_col=None):
    out=pd.DataFrame(index=df.index)
    out['title_key']=df[title_col].map(norm_text)
    if developer_col and developer_col in df.columns:
        out['developer_key']=df[developer_col].map(norm_text)
    else:
        out['developer_key']=''
    if publisher_col and publisher_col in df.columns:
        out['publisher_key']=df[publisher_col].map(norm_text)
    else:
        out['publisher_key']=''
    if year_col and year_col in df.columns:
        out['year_key']=pd.to_numeric(df[year_col],errors='coerce')
    else:
        out['year_key']=np.nan
    return out

print('Candidate key logic defined. Do not create final bridges from title-only matches.')


Candidate key logic defined. Do not create final bridges from title-only matches.


## Phase 3 acceptance criteria

- No title-only automatic joins.
- Steam AppID is the authoritative identity within Steam sources.
- Console sales remain separate from Steam ownership.
- Platform family and generation are conformed.
- Ambiguous cross-source matches require review.
- Every bridge record must retain match method and confidence.